# TrackNetV3 — clip → prediction video

Standalone notebook. Upload your fine-tuned model (`TrackNet_finetune_best.pt`) and any `.mp4` clip, and download a video with the predicted ball trajectory drawn on it.

Run the cells top to bottom. **First turn on the GPU:** Runtime → Change runtime type → GPU → Save.

### 1. Check the GPU is on

In [ ]:
!nvidia-smi -L
import torch; print('PyTorch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

### 2. Get the TrackNetV3 code

In [ ]:
!git clone -q https://github.com/qaz812345/TrackNetV3.git
%cd TrackNetV3
!pip install -q parse pycocotools

### 3. Upload your model + clip
Click **Choose Files** and select **both** your `TrackNet_finetune_best.pt` and the `.mp4` clip you want to analyze (you can multi-select).

In [ ]:
import os, glob
os.chdir('/content/TrackNetV3')
from google.colab import files
up = files.upload()        # select BOTH your .pt model and a .mp4 clip
print('uploaded:', list(up))
print('models found:', glob.glob('*.pt'))
print('clips found :', glob.glob('*.mp4'))

### 4. Run prediction → download the annotated video
Generates its own background from the clip, runs your fine-tuned detector, draws the trajectory, and downloads the result. The output uses the `mp4v` codec because Colab's OpenCV can't encode H.264.

To analyze a different clip, set `CLIP` to its filename; leave it `None` to use the first uploaded `.mp4`.

In [ ]:
import os, sys, glob, runpy, functools, torch, cv2
os.chdir('/content/TrackNetV3'); sys.path.insert(0, '/content/TrackNetV3')
sys.modules.pop('test', None)
torch.load = functools.partial(torch.load, weights_only=False)

# Colab's OpenCV can't encode H.264 -> force mp4v so the output video actually writes
_orig = cv2.VideoWriter
cv2.VideoWriter = lambda fn, fourcc, fps, sz, *a, **k: _orig(fn, cv2.VideoWriter_fourcc(*'mp4v'), fps, sz, *a, **k)

CLIP = None   # e.g. 'my_clip.mp4'; None = first uploaded .mp4

# locate the uploaded model (.pt) and clip (.mp4)
cands = glob.glob('/content/**/*.pt', recursive=True)
assert cands, 'No .pt model found - re-run the upload cell and pick your model.'
ft = sorted([c for c in cands if 'finetune' in c.lower()] or cands)[0]
clips = sorted(glob.glob('*.mp4'))
assert clips, 'No .mp4 clip found - re-run the upload cell and pick a video.'
clip = CLIP if CLIP else clips[0]
name = os.path.splitext(os.path.basename(clip))[0]
print('model:', ft)
print('clip :', clip)

sys.argv = ['predict.py', '--video_file', clip, '--tracknet_file', ft,
            '--save_dir', 'prediction', '--output_video',
            '--max_sample_num', '80', '--batch_size', '2']
runpy.run_path('predict.py', run_name='__main__')

print('prediction dir:', os.listdir('prediction'))
from google.colab import files
out = glob.glob(f'prediction/{name}.mp4')
files.download(out[0] if out else f'prediction/{name}_ball.csv')   # video if it wrote, else the CSV

### Notes
- Works on any clip and resolution — `predict.py` builds its own background from the video.
- This runs **TrackNet-only**. InpaintNet (optional trajectory in-painting) is skipped, so no pretrained download is needed.
- Colab wipes everything if the runtime disconnects — if that happens, re-run from the top and re-upload.